# Векторний пошук з SQLite (з вже створеною базою)

В цьому ноутбуці розглянуто використання векторного пошуку за допомогою SQLite. Базу даних на диску вже створено (як, див. ноутбук vector_search.ipynb). 

## Повторне підключення до бази

У новому сеансі пошуку ми можемо відкрити індекс без нового вбудовування.

In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Тепер ми можемо шукати.

In [2]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

Ми завантажуємо модель вбудовування для кодування запиту, але не вбудовуємо повторно всі документи, як у випадку використання Minsearch. 

## Інтеграція в RAG

In [4]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from google import genai

load_dotenv()
gemini_client = genai.Client()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=gemini_client,
)

Створимо запит.

In [5]:
vector_assistant.rag("the program has already begun, can I still sign up?")

/workspaces/LLM_Zoomcamp_2026/02-vector-search/rag_helper.py:77: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.llm_client.interactions.create(


'Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

Закриємо з'єднання.

In [6]:
vs_index.close()